# Recommend — Model & Language Comparison (2×3 grid)

Separate notebook from `Recommend_V3.ipynb` — V3 stays the current production
pipeline untouched. This notebook reuses V3's re-ranking/scoring logic
exactly, but swaps in two things V3 held fixed: the **embedding model** and
the **text-language condition**, so the two can be compared side by side.

**The grid: {English-only translated, native English+French untranslated} × {TF-IDF, `bge-m3`, `Qwen3-Embedding-0.6B`} = 6 runs**

- **English-only translated** — the same condition V3 already runs: catalog
  text built from the `_EN` columns (`Course Title_EN`, `Subjects_EN`,
  `Skills_EN`, `Description_EN`), i.e. every course description translated
  into English regardless of its original library.
- **Native English+French untranslated** — catalog text built from the
  *original* columns (`Course Title`, `Subjects`, `Skills`, `Description`),
  so `en-US` library rows stay in English and `fr-FR` library rows stay in
  French, exactly as authored. No translation step involved at all.

**Scope note, flagged rather than silently assumed**: only the *catalog*
side has a native-French variant. `employees_final.csv` only has `Field_EN`
— there is no French employee-side field to swap in — so employee text
(`Field_EN` + `Skills` + `Preferred_Formation`) is identical across both
language conditions. The comparison is really "does the catalog need
translating," not a full bilingual employee↔course comparison. Worth saying
explicitly in the report so it isn't overstated.

**Expected result, not a bug if it happens**: TF-IDF on the native condition
is expected to fail badly on French courses, since employee/preference text
is always English and TF-IDF only catches exact token overlap — an English
query has zero token overlap with a French course description. If TF-IDF's
scores collapse to ~0 for every `fr-FR` course under the native condition,
that's the expected, reportable finding (cross-lingual matching requires a
multilingual embedding model), not a pipeline bug.

**Also checked here**: whether the two known hub courses from V3's
hand-checking (`"Careers in Commercial Banking"` and the C-Suite leadership
course, both from Phase 19 of `PROJECT_JOURNAL.md`) still dominate
top-1 picks under `Qwen3-Embedding-0.6B` — if the same courses stay hubs
across models, that's evidence the hubness effect (Radovanović et al.) is
structural to high-dimensional embedding spaces generally, not specific to
`bge-m3`.

**Boost weights are unchanged from V3** — `FIELD_BOOST`, `PREFERENCE_BOOST`,
etc. are still the same reasoned-but-unvalidated constants from
`RECOMMENDER_PROJECT.md` §5. This notebook is about the embedding/language
choice, not weight tuning (that's still deferred to the leave-one-out step).

**Not run in this notebook.** Cells are written but not executed — same
reason as V3: no GPU/network access on the assistant's end for `bge-m3`,
`Qwen3-Embedding-0.6B`, or downloading the models. Run locally (RTX 4060
Laptop GPU) and paste back results/errors, print output, or anything that
looks wrong.

**Found and fixed while building this (worth checking against V3 too)**: a
small synthetic smoke test of this notebook's logic (3 fake employees, 4
fake courses) surfaced a real bug — on **pandas 3.x**, `.values` on a
text column now returns a `StringArray` (pandas' new default "future
string" dtype) instead of a plain numpy array, and `StringArray` doesn't
support the `[:, None] == [None, :]` broadcasting the language/level
matching relies on — it raises `ValueError: Lengths of operands do not
match`. Fixed here with a `to_plain_array()` helper (`.astype(object).to_numpy()`)
wherever a text column feeds a broadcast comparison. **`Recommend_V3.ipynb`
uses the exact same `.values` pattern** (cell 23) — worth checking what
pandas version you actually have installed (`pip show pandas`) before
running V3 again; if it's pandas 3.x, V3 has the identical latent bug and
would need the same fix. Not touching V3 to fix this without your say-so,
since it isn't yet confirmed to be broken for you (depends on your
installed pandas version) — flagging it here since this notebook needed
the fix regardless.

 `scikit-learn` (for the TF-IDF baseline —
`TfidfVectorizer` + `cosine_similarity`). Also needs
`transformers>=4.51.0` and `sentence-transformers>=2.7.0` specifically for
`Qwen3-Embedding-0.6B` (older versions raise `KeyError: 'qwen3'`) — worth
checking `pip show transformers sentence-transformers` before running if
the Qwen3 cell errors out.


## 1. Setup & config

In [ ]:
import re
import time
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

BASE = Path.cwd()
if not (BASE / "data" / "processed").is_dir():
    BASE = BASE.parent

DATA_PROCESSED = BASE / "data" / "processed"
OUTPUTS = BASE / "outputs" / "model_comparison"
OUTPUTS.mkdir(parents=True, exist_ok=True)

EMPLOYEES_PATH = DATA_PROCESSED / "employees_final.csv"
CATALOG_FULL_PATH = DATA_PROCESSED / "catalog_full_en_fr_translated.csv"
HISTORY_PATH = DATA_PROCESSED / "history_cleaned.csv"

RANDOM_SEED = 42
TOP_K = 3

def to_plain_array(series: pd.Series) -> np.ndarray:
    '''Force a plain numpy object array out of a pandas Series.

    Needed because pandas 3.x defaults to a "future string" dtype for
    object-like columns (pd.options.future.infer_string), whose .values
    returns a StringArray (a pandas ExtensionArray) rather than a plain
    ndarray. StringArray supports [:, None] reshaping but NOT full
    elementwise broadcasting against another differently-shaped
    StringArray (e.g. (n,1) == (1,m)), which V3's original .values
    pattern silently relies on for the language/level comparisons below.
    On pandas <2.x this is a no-op; on pandas 3.x it's required or
    score_and_rank/compute_field_score_norm raise
    'Lengths of operands do not match'. Confirmed via a small synthetic
    smoke test before this was written into the grid loop.
    '''
    return series.astype(object).to_numpy()




## 2. Load the three real files (identical to `Recommend_V3.ipynb`)

In [ ]:
employees = pd.read_csv(EMPLOYEES_PATH).fillna("")
catalog_full = pd.read_csv(CATALOG_FULL_PATH).fillna("")
history = pd.read_csv(HISTORY_PATH).fillna("")

assert employees["Email"].is_unique, "Employee emails should be unique"
print("employees:", employees.shape)
print("catalog_full:", catalog_full.shape)
print("history:", history.shape)


## 3. Derive the active-only candidate pool

Identical filters to V3: `Status == "active"`, drop `Level == "general"`.
Language-invariant columns (`_level_norm`, `_lang`) are computed once here
and reused across every run in the grid — only the *text* used for
embedding/field-matching differs by language condition.

In [ ]:
n_before = len(catalog_full)
catalog_pool = catalog_full[
    (catalog_full["Status"].str.lower() == "active")
    & (catalog_full["Level"].str.lower() != "general")
].copy().reset_index(drop=True)

catalog_pool["_level_norm"] = catalog_pool["Level"].str.lower().str.strip()
catalog_pool["_lang"] = catalog_pool["Library"].str.upper().map({"EN-US": "EN", "FR-FR": "FR"}).fillna("")

print(f"Full catalog: {n_before} -> candidate pool: {len(catalog_pool)} rows "
      f"(active only, general-level excluded)")
print("Library breakdown of the pool:")
print(catalog_pool["_lang"].value_counts())


## 4. Join history to employees, derive per-employee signals (identical to V3)

In [ ]:
LANGUAGE_THRESHOLD = 0.6

id_to_library = dict(zip(catalog_full["Course ID"].astype(str), catalog_full["Library"]))

history["_course_id"] = history["ID de contenu"].astype(str)
history["_status_norm"] = history["Status"].str.strip().str.lower()

def derive_employee_signals(group):
    completed = group[group["_status_norm"] == "completed"]
    in_progress = group[group["_status_norm"] == "in-progress"]

    excluded_ids = set(completed["_course_id"]).union(set(in_progress["_course_id"]))

    langs = completed["_course_id"].map(id_to_library).dropna()
    inferred_language = ""
    if len(langs) > 0:
        counts = Counter(langs)
        top_lang, top_count = counts.most_common(1)[0]
        if top_count / len(langs) > LANGUAGE_THRESHOLD:
            inferred_language = "EN" if top_lang == "en-US" else "FR" if top_lang == "fr-FR" else ""

    return pd.Series({
        "Excluded_Course_IDs": excluded_ids,
        "Inferred_Language": inferred_language,
    })

emp_signals = history.groupby("email_n").apply(derive_employee_signals)
employees = employees.merge(emp_signals, left_on="Email", right_index=True, how="left")
employees["Excluded_Course_IDs"] = employees["Excluded_Course_IDs"].apply(lambda x: x if isinstance(x, set) else set())
employees["Inferred_Language"] = employees["Inferred_Language"].fillna("")

print("Employees with an inferred language:", (employees["Inferred_Language"] != "").sum())
print("Employees with any excluded courses:", employees["Excluded_Course_IDs"].apply(len).gt(0).sum())


## 5. Build text for both language conditions

- **Employee text** (`_text`): unchanged from V3, English-only in both
  conditions — see the scope note in the intro cell.
- **Course text**: two variants —
  - `_text_translated_en` — same as V3, built from the `_EN` columns.
  - `_text_native` — built from the original, untranslated columns; `en-US`
    rows stay English, `fr-FR` rows stay French, exactly as authored.

In [ ]:
employees["_text"] = (
    employees["Field_EN"].astype(str) + ". "
    + employees["Skills"].astype(str) + ". "
    + employees["Preferred_Formation"].astype(str)
).str.strip()

catalog_pool["_text_translated_en"] = (
    catalog_pool["Course Title_EN"].astype(str) + ". "
    + catalog_pool["Subjects_EN"].astype(str) + ". "
    + catalog_pool["Skills_EN"].astype(str) + ". "
    + catalog_pool["Description_EN"].astype(str)
).str.strip()

catalog_pool["_text_native"] = (
    catalog_pool["Course Title"].astype(str) + ". "
    + catalog_pool["Subjects"].astype(str) + ". "
    + catalog_pool["Skills"].astype(str) + ". "
    + catalog_pool["Description"].astype(str)
).str.strip()

TEXT_COLUMNS = {
    "translated_en": {"course_text": "_text_translated_en", "skills_col": "Skills_EN", "subjects_col": "Subjects_EN"},
    "native":        {"course_text": "_text_native",        "skills_col": "Skills",    "subjects_col": "Subjects"},
}

catalog_pool[["Course ID", "_lang", "_text_translated_en", "_text_native"]].head()


## 6. Preference vocabulary gate (identical mechanism to V3, built per language condition)

Same deterministic gate as V3 §5b — a stated preference only gets embedded/
boosted if it literally contains a multi-word phrase from the catalog's own
vocabulary. Built as a function here since the vocabulary itself depends on
which language condition's `Skills`/`Subjects` columns are in play (the
native condition's vocabulary includes French phrases the translated
condition doesn't).

In [ ]:
PREFERENCE_VOCAB_STOPWORDS = {"and", "of", "the", "for", "with", "your", "from", "using", "into"}
PREFERENCE_VOCAB_BLOCKLIST = {"leadership skills", "professional development"}  # confirmed too-generic in V3

def build_preference_vocab(catalog: pd.DataFrame, skills_col: str, subjects_col: str) -> set:
    phrases = set()
    for col in [skills_col, subjects_col]:
        for val in catalog[col].dropna():
            for term in str(val).split(","):
                term = term.strip().lower()
                if len(term.split()) >= 2 and term not in PREFERENCE_VOCAB_BLOCKLIST:
                    phrases.add(term)
    return phrases

def preference_is_informative(text: str, vocab: set) -> bool:
    cleaned = str(text).strip().lower()
    if cleaned == "":
        return False
    return any(phrase in cleaned for phrase in vocab)

def compute_has_pref(catalog: pd.DataFrame, skills_col: str, subjects_col: str) -> np.ndarray:
    vocab = build_preference_vocab(catalog, skills_col, subjects_col)
    has_pref = employees["Preferred_Formation"].apply(lambda t: preference_is_informative(t, vocab)).values
    print(f"  vocabulary: {len(vocab)} multi-word phrases -> "
          f"{has_pref.sum()} / {len(employees)} employees have an informative preference")
    return has_pref


## 7. Boost weights and field-match helper (unchanged from V3)

Weight *magnitudes* are explicitly not being tuned here — see
`WHATS_LEFT.md` §1, deferred to after the leave-one-out evaluation exists.

In [ ]:
FIELD_BOOST = 0.03
PREFERENCE_BOOST = 0.15
LANGUAGE_MATCH_BOOST = 0.07
LANGUAGE_DEFAULT_EN_BOOST = 0.03
LEVEL_MATCH_BOOST = 0.05
LEVEL_MISMATCH_PENALTY = 0.10
TRAINING_TYPE_BOOST = 0.05
DOMAIN_GATE_PENALTY = 0.08

FIELD_STOPWORDS = {"and", "of", "the", "for", "role", "focus"}
MIN_FIELD_TOKEN_LEN = 4

def strip_accents(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))

def field_tokens(bu_new: str) -> list:
    raw = re.split(r"[^a-zA-Z]+", strip_accents(bu_new).lower())
    return [t for t in raw if len(t) >= MIN_FIELD_TOKEN_LEN and t not in FIELD_STOPWORDS]

KIND_SENTENCES = {
    "field": "it matches your field",
    "pref_strong": "it matches what you said you want to learn",
    "pref_weak": "it's related to what you said you want to learn",
    "lang_hist": "it's in the language you've mostly trained in",
    "lang_default": "it's offered in English by default",
    "level_match": "it fits your seniority level",
    "level_mismatch": "it's a different level than yours",
    "type_match": "it matches your preferred training format",
    "no_domain": "it has no clear connection to your field or stated interest",
}
STRONG_KINDS = {"field", "pref_strong", "pref_weak", "type_match"}

def sentence_for(items):
    phrases = [KIND_SENTENCES[k] for k, _, _ in items]
    if not phrases:
        return ""
    if len(phrases) == 1:
        return phrases[0]
    return ", ".join(phrases[:-1]) + " and " + phrases[-1]


## 8. Field match — TF-IDF-weighted token overlap, parameterized by course text column

Identical mechanism to V3 §9 step 1a, wrapped as a function so it can be
recomputed against either `_text_translated_en` or `_text_native` (the two
texts tokenize differently — an English `Bu_new` token like `"finance"`
naturally won't appear in a French course description under the native
condition, which is itself a real, reportable effect of that condition, not
a bug to fix).

In [ ]:
def compute_field_score_norm(course_text_col: str) -> np.ndarray:
    unique_bu = employees["Bu_new"].unique()
    bu_tokens = {bu: set(field_tokens(bu)) for bu in unique_bu}
    all_tokens = sorted(set().union(*bu_tokens.values())) if len(bu_tokens) else []
    token_index = {t: i for i, t in enumerate(all_tokens)}

    course_text_norm = catalog_pool[course_text_col].map(lambda t: strip_accents(t).lower())
    n_course = len(catalog_pool)
    course_token_matrix = np.zeros((n_course, len(all_tokens)), dtype=bool)
    for tok, col in token_index.items():
        pattern = re.compile(r"\b" + re.escape(tok) + r"\b")
        course_token_matrix[:, col] = course_text_norm.map(lambda t: bool(pattern.search(t))).values

    doc_freq = course_token_matrix.sum(axis=0)
    token_idf = np.log((n_course + 1) / (doc_freq + 1)) + 1

    n_emp = len(employees)
    emp_token_weights = np.zeros((n_emp, len(all_tokens)), dtype=np.float32)
    for i, bu in enumerate(to_plain_array(employees["Bu_new"])):
        for tok in bu_tokens[bu]:
            emp_token_weights[i, token_index[tok]] = token_idf[token_index[tok]]

    raw_field_score = emp_token_weights @ course_token_matrix.astype(np.float32).T
    emp_max_score = emp_token_weights.sum(axis=1, keepdims=True)
    emp_max_score_safe = np.where(emp_max_score == 0, 1.0, emp_max_score)
    field_score_norm = np.clip(raw_field_score / emp_max_score_safe, 0.0, 1.0)
    return field_score_norm


## 9. Embedding backends — one function per model, same (sim_matrix, pref_sim_matrix) contract

All three return `(sim_matrix, pref_sim_matrix)` of shape `(n_emp, n_course)`
so the ranking step downstream (§10) doesn't need to know which model
produced them.

- **TF-IDF**: `TfidfVectorizer` fit once on employee+course text so the
  vocabulary/IDF stats come from the actual corpus, then `cosine_similarity`
  for both the employee×course and preference×course matrices.
- **`bge-m3`**: unchanged from V3 — symmetric model, no query/document
  prompt distinction.
- **`Qwen3-Embedding-0.6B`**: per the model card, queries benefit from a
  `prompt_name="query"` prefix while documents don't — employee text and
  preference text are the "query" side here (an employee looking for a
  course), course text is the "document" side. Requires
  `transformers>=4.51.0` / `sentence-transformers>=2.7.0`.

In [ ]:
def embed_tfidf(emp_texts, course_texts, pref_texts, has_pref):
    vectorizer = TfidfVectorizer()
    corpus = list(emp_texts) + list(course_texts)
    vectorizer.fit(corpus)

    emp_tfidf = vectorizer.transform(emp_texts)
    course_tfidf = vectorizer.transform(course_texts)
    sim_matrix = cosine_similarity(emp_tfidf, course_tfidf).astype(np.float32)

    pref_texts_safe = [t if h else "" for t, h in zip(pref_texts, has_pref)]
    pref_tfidf = vectorizer.transform(pref_texts_safe)
    pref_sim_matrix = cosine_similarity(pref_tfidf, course_tfidf).astype(np.float32)

    return sim_matrix, pref_sim_matrix


def embed_sentence_transformer(model_name, emp_texts, course_texts, pref_texts, has_pref, query_prompt_name=None):
    import torch
    from sentence_transformers import SentenceTransformer, util

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  torch.cuda.is_available(): {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("  WARNING: no CUDA GPU detected -- running on CPU, this will be slow for 1,059x10,765.")

    model = SentenceTransformer(model_name, device=device)

    emp_kwargs = {"prompt_name": query_prompt_name} if query_prompt_name else {}
    emp_vectors = model.encode(
        list(emp_texts), batch_size=64, normalize_embeddings=True,
        show_progress_bar=True, **emp_kwargs,
    )
    course_vectors = model.encode(
        list(course_texts), batch_size=64, normalize_embeddings=True, show_progress_bar=True,
    )

    pref_vectors = np.zeros((len(emp_texts), emp_vectors.shape[1]), dtype=np.float32)
    has_pref = np.asarray(has_pref)
    if has_pref.any():
        pref_subset = [t for t, h in zip(pref_texts, has_pref) if h]
        pref_kwargs = {"prompt_name": query_prompt_name} if query_prompt_name else {}
        encoded = model.encode(
            pref_subset, batch_size=64, normalize_embeddings=True, show_progress_bar=True, **pref_kwargs,
        )
        pref_vectors[has_pref] = encoded

    sim_matrix = util.cos_sim(emp_vectors, course_vectors).numpy()
    pref_sim_matrix = util.cos_sim(pref_vectors, course_vectors).numpy()
    return sim_matrix, pref_sim_matrix


EMBEDDERS = {
    "tfidf": lambda emp_t, course_t, pref_t, has_pref: embed_tfidf(emp_t, course_t, pref_t, has_pref),
    "bge-m3": lambda emp_t, course_t, pref_t, has_pref: embed_sentence_transformer(
        "BAAI/bge-m3", emp_t, course_t, pref_t, has_pref, query_prompt_name=None,
    ),
    "qwen3-embedding-0.6b": lambda emp_t, course_t, pref_t, has_pref: embed_sentence_transformer(
        "Qwen/Qwen3-Embedding-0.6B", emp_t, course_t, pref_t, has_pref, query_prompt_name="query",
    ),
}


## 10. Scoring & ranking (identical mechanism to V3 §9) — wrapped as one function

Same formulas as `Recommend_V3.ipynb` cells 23-25: field/preference/
language/level/training-type boosts, the domain gate, exclusion masking,
`np.argpartition` top-3 selection, and `reason`/`explanation`/
`flagged_for_review` reconstruction. Nothing here is new logic — it's V3's
scoring step made callable so it can run 6 times against different
`(sim_matrix, pref_sim_matrix, field_score_norm)` inputs.

In [ ]:
def score_and_rank(sim_matrix, pref_sim_matrix, has_pref, field_score_norm):
    n_emp, n_course = sim_matrix.shape

    row_min = pref_sim_matrix.min(axis=1, keepdims=True)
    row_max = pref_sim_matrix.max(axis=1, keepdims=True)
    row_range_safe = np.where((row_max - row_min) == 0, 1.0, row_max - row_min)
    pref_norm = (pref_sim_matrix - row_min) / row_range_safe
    pref_contribution = np.where(has_pref[:, None], PREFERENCE_BOOST * pref_norm, 0.0)

    emp_lang = to_plain_array(employees["Inferred_Language"])
    course_lang = to_plain_array(catalog_pool["_lang"])
    has_lang_pref = (emp_lang != "")[:, None]
    lang_match = has_lang_pref & (emp_lang[:, None] == course_lang[None, :])
    lang_default = (~has_lang_pref) & (course_lang[None, :] == "EN")
    lang_boost = np.where(lang_match, LANGUAGE_MATCH_BOOST, np.where(lang_default, LANGUAGE_DEFAULT_EN_BOOST, 0.0))

    emp_level = to_plain_array(employees["Employee_Level"])
    course_level = to_plain_array(catalog_pool["_level_norm"])
    course_has_level = (course_level != "")[None, :]
    level_match = course_has_level & (course_level[None, :] == emp_level[:, None])
    level_mismatch = course_has_level & (course_level[None, :] != emp_level[:, None])
    level_boost = np.where(level_match, LEVEL_MATCH_BOOST, np.where(level_mismatch, -LEVEL_MISMATCH_PENALTY, 0.0))

    pref_type = to_plain_array(employees["Preferred_Training_Type"].astype(str).str.strip())
    training_boost = np.where((pref_type == "Online")[:, None], TRAINING_TYPE_BOOST, 0.0)

    field_boost = FIELD_BOOST * field_score_norm

    no_domain_signal = (field_score_norm == 0) & (pref_contribution == 0)
    domain_gate_boost = np.where(no_domain_signal, -DOMAIN_GATE_PENALTY, 0.0)

    rank_score_matrix = sim_matrix + field_boost + pref_contribution + lang_boost + level_boost + training_boost + domain_gate_boost

    course_id_index = {str(cid): idx for idx, cid in enumerate(catalog_pool["Course ID"])}
    exclusion_mask = np.zeros((n_emp, n_course), dtype=bool)
    for i, excluded in enumerate(employees["Excluded_Course_IDs"]):
        for cid in excluded:
            col = course_id_index.get(cid)
            if col is not None:
                exclusion_mask[i, col] = True

    rank_score_matrix_masked = np.where(exclusion_mask, -np.inf, rank_score_matrix)

    k = min(TOP_K, n_course)
    top_idx_unsorted = np.argpartition(-rank_score_matrix_masked, k - 1, axis=1)[:, :k]
    row_indices = np.arange(n_emp)[:, None]
    top_scores_unsorted = rank_score_matrix_masked[row_indices, top_idx_unsorted]
    order = np.argsort(-top_scores_unsorted, axis=1)
    top_idx = np.take_along_axis(top_idx_unsorted, order, axis=1)

    def describe_pick(i, j, title_col):
        course = catalog_pool.iloc[j]
        boosts = []

        field_val = float(field_boost[i, j])
        if field_val > 0:
            boosts.append(("field", "field match", field_val))

        if has_pref[i]:
            norm_val = float(pref_norm[i, j])
            contribution = float(pref_contribution[i, j])
            raw_sim = float(pref_sim_matrix[i, j])
            if norm_val > 0.7:
                boosts.append(("pref_strong", f"matches stated interest ({raw_sim:.2f})", contribution))
            elif norm_val > 0.3:
                boosts.append(("pref_weak", f"related to stated interest ({raw_sim:.2f})", contribution))
            elif contribution > 0:
                boosts.append((None, "__silent__", contribution))

        if lang_match[i, j]:
            boosts.append(("lang_hist", "language match (from history)", LANGUAGE_MATCH_BOOST))
        elif lang_default[i, j]:
            boosts.append(("lang_default", "default English", LANGUAGE_DEFAULT_EN_BOOST))

        if level_match[i, j]:
            boosts.append(("level_match", "level match", LEVEL_MATCH_BOOST))
        elif level_mismatch[i, j]:
            boosts.append(("level_mismatch", "level mismatch", -LEVEL_MISMATCH_PENALTY))

        if pref_type[i] == "Online":
            boosts.append(("type_match", "training type match", TRAINING_TYPE_BOOST))

        if no_domain_signal[i, j]:
            boosts.append(("no_domain", "no clear connection to your field or stated interest", -DOMAIN_GATE_PENALTY))

        visible = [(k_, l, v) for k_, l, v in boosts if k_ is not None]
        visible_sorted = sorted(visible, key=lambda x: -abs(x[2]))
        numeric_reason = ", ".join(f"{l} ({v:+.2f})" for _, l, v in visible_sorted)

        positive = [x for x in visible_sorted if x[2] > 0]
        negative = [x for x in visible_sorted if x[2] < 0]
        pos_sentence = sentence_for(positive[:3])
        neg_sentence = sentence_for(negative)
        natural = f"Recommended because {pos_sentence}." if pos_sentence else "Recommended based on overall profile similarity."
        if neg_sentence:
            natural += f" Note: {neg_sentence}."

        flagged = not any(k_ in STRONG_KINDS for k_, _, _ in positive)

        return {
            "Course_ID": course["Course ID"],
            "Course_Title": course[title_col],
            "similarity": float(sim_matrix[i, j]),
            "rank_score": float(rank_score_matrix_masked[i, j]),
            "reason": numeric_reason,
            "explanation": natural,
            "flagged_for_review": flagged,
        }

    title_col = "Course Title_EN" if "Course Title_EN" in catalog_pool.columns else "Course Title"

    results = []
    for i, emp in employees.iterrows():
        if rank_score_matrix_masked[i].max() == -np.inf:
            results.append({
                "Employee_ID": emp["Employee_ID"], "Email": emp["Email"], "Rank": None,
                "Course_ID": None, "Course_Title": None, "similarity": None, "rank_score": None,
                "reason": "no eligible courses remaining", "explanation": "no eligible courses remaining",
                "flagged_for_review": True,
            })
            continue
        for rank, j in enumerate(top_idx[i], start=1):
            rec = describe_pick(i, j, title_col)
            results.append({
                "Employee_ID": emp["Employee_ID"], "Email": emp["Email"], "Rank": rank,
                "Course_ID": rec["Course_ID"], "Course_Title": rec["Course_Title"],
                "similarity": round(rec["similarity"], 4), "rank_score": round(rec["rank_score"], 4),
                "reason": rec["reason"], "explanation": rec["explanation"],
                "flagged_for_review": rec["flagged_for_review"],
            })

    return pd.DataFrame(results)


## 11. Hub-course check

Reports the most frequently recommended (top-1) courses for a run, and
specifically looks up the two courses flagged as hubs during V3's hand-
checking (Phase 19) — `"Careers in Commercial Banking"` and the C-Suite
leadership course — by title match (robust to exact ID differences), so we
can see directly whether they still dominate under a different model.

In [ ]:
KNOWN_HUB_TITLE_FRAGMENTS = ["commercial banking", "c-suite"]

def hub_course_report(results_df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    top1 = results_df[results_df["Rank"] == 1]
    counts = top1["Course_Title"].value_counts().head(top_n)
    report = counts.rename("times_recommended_as_top1").to_frame()
    report["pct_of_employees"] = (report["times_recommended_as_top1"] / len(top1) * 100).round(1)
    return report

def known_hub_frequency(results_df: pd.DataFrame) -> dict:
    top1 = results_df[results_df["Rank"] == 1]
    out = {}
    for fragment in KNOWN_HUB_TITLE_FRAGMENTS:
        mask = top1["Course_Title"].str.lower().str.contains(fragment, na=False)
        out[fragment] = int(mask.sum())
    return out


## 12. Run the full 2×3 grid

For each of the 2 language conditions, the preference vocabulary and
field-match scores are recomputed once (they don't depend on the embedding
model), then all 3 embedding backends are run against that condition's
text. Each run's results are saved to its own CSV under
`outputs/model_comparison/`.

This will take a while locally — `bge-m3` and `Qwen3-Embedding-0.6B` both
need to embed ~1,059 employees + ~10,765 courses (+ preferences) **twice
each** (once per language condition), on top of the two TF-IDF runs which
are comparatively instant.

In [ ]:
LANGUAGE_CONDITIONS = ["translated_en", "native"]
MODELS = ["tfidf", "bge-m3", "qwen3-embedding-0.6b"]

all_results = {}     # (lang_condition, model) -> results_df
run_summary = []      # one row per run, for the comparison table in section 13

for lang_condition in LANGUAGE_CONDITIONS:
    cols = TEXT_COLUMNS[lang_condition]
    print("=" * 80)
    print(f"LANGUAGE CONDITION: {lang_condition}  (course text: {cols['course_text']})")
    print("=" * 80)

    print("Building preference vocabulary + informativeness gate...")
    has_pref = compute_has_pref(catalog_pool, cols["skills_col"], cols["subjects_col"])

    print("Computing field-match scores...")
    field_score_norm = compute_field_score_norm(cols["course_text"])
    print(f"  field boost fires for {(field_score_norm > 0).mean():.1%} of (employee, course) pairs")

    emp_texts = employees["_text"].tolist()
    course_texts = catalog_pool[cols["course_text"]].tolist()
    pref_texts = employees["Preferred_Formation"].tolist()

    for model_name in MODELS:
        print("-" * 80)
        print(f"MODEL: {model_name}  |  LANGUAGE: {lang_condition}")
        t0 = time.time()

        sim_matrix, pref_sim_matrix = EMBEDDERS[model_name](emp_texts, course_texts, pref_texts, has_pref)
        results_df = score_and_rank(sim_matrix, pref_sim_matrix, has_pref, field_score_norm)

        elapsed = time.time() - t0
        out_path = OUTPUTS / f"recommendations_{lang_condition}_{model_name}.csv"
        results_df.to_csv(out_path, index=False)

        top1 = results_df[results_df["Rank"] == 1]
        flagged_rate = top1["flagged_for_review"].mean()
        mean_sim = top1["similarity"].mean()
        hub_freq = known_hub_frequency(results_df)

        run_summary.append({
            "language_condition": lang_condition,
            "model": model_name,
            "elapsed_sec": round(elapsed, 1),
            "employees_scored": len(top1),
            "flagged_for_review_rate": round(flagged_rate, 4),
            "mean_top1_similarity": round(mean_sim, 4),
            "commercial_banking_top1_count": hub_freq["commercial banking"],
            "c_suite_top1_count": hub_freq["c-suite"],
        })

        all_results[(lang_condition, model_name)] = results_df
        print(f"  done in {elapsed:.1f}s -- saved to {out_path}")
        print(f"  flagged_for_review rate: {flagged_rate:.1%}, mean top-1 similarity: {mean_sim:.3f}")
        print(f"  known hub courses in top-1 picks: {hub_freq}")

print("=" * 80)
print("Grid complete.")


## 13. Comparison summary across all 6 runs

The core comparison table for deciding which model/language combination to
carry forward. Also prints each run's top-10 most-recommended courses so
hub persistence (or the lack of it) is visible directly, not just via the
two known hub titles checked automatically above.

In [ ]:
summary_df = pd.DataFrame(run_summary)
summary_path = OUTPUTS / "comparison_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved comparison summary to {summary_path}")
summary_df


In [ ]:
for (lang_condition, model_name), results_df in all_results.items():
    print("=" * 80)
    print(f"Top-10 most-recommended (top-1) courses -- {model_name} / {lang_condition}")
    print(hub_course_report(results_df))


## 14. What to look at once this comes back

- **`comparison_summary.csv`** — `flagged_for_review_rate` and
  `mean_top1_similarity` side by side across all 6 runs. Also check
  `elapsed_sec` for a rough sense of `Qwen3-Embedding-0.6B`'s cost relative
  to `bge-m3` on the same GPU.
- **TF-IDF / native row** — expected to show near-zero `mean_top1_similarity`
  and/or a `flagged_for_review_rate` near 100% for employees whose likely
  best-fit courses are French, since English employee text has no token
  overlap with French course text under TF-IDF. If this shows up, it's the
  expected result, not a bug (see the intro cell).
- **`commercial_banking_top1_count` / `c_suite_top1_count`** — if these stay
  nonzero (or similarly high) across all three models, that's evidence the
  hubness effect is structural to the embedding-similarity approach itself,
  not specific to `bge-m3`. If they drop to ~0 under `Qwen3-Embedding-0.6B`,
  that's evidence it's more model-specific than structural — genuinely
  informative either way, worth reporting as-is.
- **Per-run top-10 tables (section 13)** — worth a manual glance for *new*
  hub courses that weren't visible in V3's hand-checking (V3 only ever ran
  `bge-m3` / `translated_en`).
